In [ ]:
# INSERT HERE colab stuff (tensorflow_version, mount drive, etc.)

In [ ]:
# example with shakespeare data
!python3 prepare_data2.py shakespeare_input.txt shake \\n\\n -m 500

In [ ]:
import pickle

import tensorflow as tf
from prepare_data2 import parse_seq


# data
bs = 128
data = tf.data.TFRecordDataset("shake.tfrecords")
data = data.map(parse_seq)
data = data.shuffle(46000).padded_batch(bs, (-1,), drop_remainder=True).repeat()

vocab = pickle.load(open("shake_vocab", mode="rb"))
vocab_size = len(vocab)
ind_to_ch = {ind: ch for (ch, ind) in vocab.items()}

print(vocab_size)

In [ ]:
print(ind_to_ch)

In [ ]:
# if you wanna look at the data: this will print everything forever
for ind, thing in enumerate(data):
    inds = thing[0].numpy()
    to_chars = "".join([ind_to_ch[ind] for ind in inds])
    print(ind)
    print(to_chars)
    print(len(inds))
    print()

In [ ]:
one_hotter = tf.keras.layers.Lambda(lambda x: tf.one_hot(x, depth=vocab_size))


lstm1 = tf.keras.layers.LSTM(512, return_sequences=True, stateful=True)
lstm2 = tf.keras.layers.LSTM(512, return_sequences=True, stateful=True)


out = tf.keras.layers.Dense(vocab_size)
model = tf.keras.Sequential([lstm1, lstm2, out])
model.build([bs, None, vocab_size])

In [ ]:
# alternative with the functional API
inp = tf.keras.layers.Input((None,))

# ... and embeddings instead of one-hot
embedding = tf.keras.layers.Embedding(vocab_size, 20)(inp)
lstm1 = tf.keras.layers.LSTM(512, return_sequences=True, stateful=True)(embedding)
lstm2 = tf.keras.layers.LSTM(512, return_sequences=True, stateful=True)(lstm1)
out = tf.keras.layers.Dense(vocab_size)(lstm2)

model = tf.keras.Model(inp, out)

In [ ]:
# training
steps = 100*35000 // bs
opt = tf.optimizers.Adam()


def run_rnn_on_seq(seq_batch):  # batch x time
    print("retrace")
    model.reset_states()
    
    # there are other ways to do the mask, e.g. using tf.not_equal
    # or tf.where
    count_actual = tf.math.count_nonzero(seq_batch, axis=-1) - 1
    mask = tf.sequence_mask(count_actual, dtype=tf.float32)
    with tf.GradientTape() as tape:
        # if you use embeddings, be sure to remove the one_hotter here
        out_seq = model(one_hotter(seq_batch[:, :-1]))
        loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=out_seq, labels=seq_batch[:, 1:])
        loss = loss*mask
        loss = tf.reduce_sum(loss) / tf.reduce_sum(tf.cast(count_actual, tf.float32))

    grads = tape.gradient(loss, model.trainable_variables)
    #glob_norm = tf.linalg.global_norm(grads)
    #grads = [g/glob_norm for g in grads]
    opt.apply_gradients(zip(grads, model.trainable_variables))

    return loss


# do this instead of tf.function decorator
# to avoid trouble with variable sequence lengths
input_shape = tf.TensorSpec([None, None], dtype=tf.int32)
run_rnn_on_seq = tf.function(run_rnn_on_seq, input_signature=(input_shape,))

In [ ]:
# alternative ways to do mask
#mask = tf.where(tf.equal(seq_batch, 0), tf.zeros_like(seq_batch), tf.ones_like(seq_batch))
#mask = tf.cast(tf.not_equal(seq_batch, tf.zeros_like(seq_batch)), tf.float32)

In [ ]:
for step, seqs in enumerate(data):
    xent_avg = run_rnn_on_seq(seqs)

    if not step % 200:
        print("Step: {} Loss: {}".format(step, xent_avg))

    if step > steps:
        break

In [ ]:
import numpy as np


def up_to_end(seq):
    new = []
    for ind in seq:
        new.append(ind)
        if ind == 2:
            return new
    return new


# sampling is kinda dumb: generate a fixed sequence length and then
# throw away everything after </S>
# could also just stop sampling directly when </S> is generated
# but this is not straightforward in the batch generation case like here
# note that we generate 128 sequences in parallel because the stateful RNN
# _always_ needs 128 as batch size
def sample(n_steps):
    model.reset_states()
    gen = np.ones([128, 1], dtype=np.int32)

    for step in range(n_steps):
        probs = tf.nn.softmax(model(one_hotter(gen[:, -1:]))).numpy()
        #last = np.argmax(probs, axis=-1)
        last = np.array([np.random.choice(vocab_size, p=ps.squeeze()) for ps in probs])[:, np.newaxis]
        gen = np.concatenate([gen, last], axis=1)
    return ["".join([ind_to_ch[ind] for ind in up_to_end(seq)]) for seq in gen]
        
agg = sample(500)

In [ ]:
for ind, seq in enumerate(agg):
    print("Sequence", ind)
    print(seq)
    print()